# Guía Desde Cero – Parcial 2 Práctico
## ISIS-2611 | Teoría + Ejercicios de Corrección

---

Esta guía explica **primero la teoría** de cada concepto y luego muestra un ejercicio de código con errores para que lo corrijas.

**Cómo usarla:**
1. Lee la teoría con calma.
2. Mira el código con errores e intenta identificar qué está mal antes de leer la solución.
3. Lee la solución y el por qué del error.

---

## Índice
| Sección | Tema |
|---------|------|
| 1 | Tipos de problemas de ML |
| 2 | Funciones de activación |
| 3 | Funciones de pérdida (loss) |
| 4 | El split de datos y Data Leakage |
| 5 | Preprocesamiento: escalado, imputación, encoders |
| 6 | MLP – Redes neuronales densas |
| 7 | CNN – Redes convolucionales para imágenes |
| 8 | Embeddings y NLP |
| 9 | RNN, LSTM y GRU |
| 10 | Clustering |
| 11 | Naive Bayes |
| 12 | Métricas de evaluación |
| 13 | Caso completo con múltiples errores |

---
---
# SECCIÓN 1 – Tipos de problemas de Machine Learning

---

## ¿Qué tipo de problema tengo?

La primera pregunta que debes responder al leer un caso es: **¿de qué tipo de problema se trata?**
De eso depende todo: el modelo, la activación, la pérdida y las métricas.

### Paso 1: ¿Hay etiquetas (Y conocidas)?

| Respuesta | Tipo |
|-----------|------|
| **Sí** – el dataset tiene una columna que queremos predecir | **Aprendizaje Supervisado** |
| **No** – no hay columna objetivo, solo queremos agrupar | **Aprendizaje No Supervisado** |

### Paso 2: Si es supervisado, ¿qué tipo de salida?

| La salida es... | Tipo | Ejemplo |
|----------------|------|--------|
| Un número continuo | **Regresión** | Precio de una casa, temperatura |
| Una categoría entre 2 opciones | **Clasificación binaria** | Spam/No spam, Fraude/No fraude |
| Una categoría entre N > 2 opciones | **Clasificación multiclase** | Tipo de flor (3), dígito (10), ticket (5) |

### Paso 3: Si es no supervisado, ¿qué queremos?

| Quiero... | Algoritmo |
|-----------|----------|
| Grupos de forma esférica en datos numéricos | **K-Means** |
| Grupos de forma arbitraria, con outliers | **DBSCAN** |
| Ver una jerarquía de grupos (dendrograma) | **Jerárquico aglomerativo** |

### Ejercicio 1 – Identifica el tipo de problema

Lee cada caso y responde antes de ver la solución.

**Caso A:** "Queremos clasificar reseñas de películas como positiva, negativa o neutral."

**Caso B:** "Dado el historial de compras, queremos predecir cuánto va a gastar el cliente el próximo mes."

**Caso C:** "Tenemos fotos de galaxias sin categorías y queremos encontrar grupos similares."

**Caso D:** "Un banco quiere predecir si un cliente va a pagar o no va a pagar un crédito."

**Caso E:** "Queremos agrupar correos sin ninguna etiqueta previa, considerando que algunos correos no pertenecen a ningún grupo."

**✅ RESPUESTAS:**

| Caso | Tipo | Razón |
|------|------|-------|
| A | Clasificación **multiclase** (3 clases) | Hay etiquetas: positiva / negativa / neutral |
| B | **Regresión** | La salida es un número continuo (cuánto dinero) |
| C | **Clustering no supervisado** | No hay etiquetas. Forma libre → K-Means o DBSCAN |
| D | Clasificación **binaria** | Paga (1) o no paga (0) |
| E | **DBSCAN** | No supervisado y explícitamente mencionan "algunos no pertenecen a ningún grupo" → outliers → DBSCAN |

---
---
# SECCIÓN 2 – Funciones de Activación

---

## ¿Qué es una función de activación?

Una neurona hace una operación lineal (multiplica pesos por entradas y suma un bias).
La función de activación convierte ese resultado en algo útil.
**Sin activación**, apilar capas no sirve de nada: todo sería solo una operación lineal.

## Las 4 activaciones que necesitas saber

### `relu` – Rectified Linear Unit
```
f(x) = max(0, x)
```
- Convierte negativos en 0, positivos los deja igual.
- **Úsala en: capas OCULTAS** de cualquier red.
- ❌ Nunca en la capa de salida de clasificación (puede dar valores > 1).

### `sigmoid`
```
f(x) = 1 / (1 + e^(-x))    →   salida en (0, 1)
```
- Aplasta cualquier valor al rango (0, 1).
- Interpreta el resultado como **probabilidad**.
- **Úsala en: capa de salida de clasificación BINARIA** (1 sola neurona).

### `softmax`
```
f(xi) = e^xi / Σ(e^xj)    →   suma de todas las salidas = 1
```
- Convierte N valores en N probabilidades que suman 1.
- **Úsala en: capa de salida de clasificación MULTICLASE** (N neuronas).
- ❌ Nunca usar sigmoid para multiclase (cada neurona sería independiente, no suman 1).

### Sin activación (lineal)
```
f(x) = x
```
- El valor sale tal cual.
- **Úsala en: capa de salida de REGRESIÓN**.

## Tabla resumen

| Tarea | Neuronas de salida | Activación de salida |
|-------|-------------------|--------------------|
| Regresión | 1 | ninguna / `linear` |
| Clasificación **binaria** | 1 | `sigmoid` |
| Clasificación **multiclase** | N (una por clase) | `softmax` |
| Capas **ocultas** (todas) | cualquier número | `relu` |

### Ejercicio 2 – Activaciones incorrectas

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: Clasificar correos en 3 categorías: trabajo, personal, spam.
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(128, activation='sigmoid', input_shape=(500,)),  # Error 1
    layers.Dense(64,  activation='sigmoid'),                       # Error 1
    layers.Dense(3,   activation='relu')                           # Error 2
])
# ¿Encuentras los 2 errores?

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: sigmoid en capas OCULTAS
#   sigmoid satura (valores cercanos a 0 o 1) y causa vanishing gradient.
#   En capas ocultas siempre relu (o variantes: leaky_relu, elu).
#
# ERROR 2: relu en la capa de SALIDA MULTICLASE (3 clases)
#   relu puede producir 0 en muchas neuronas y no genera probabilidades.
#   Para 3 clases → softmax.

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(500,)),  # ✅ relu en ocultas
    layers.Dense(64,  activation='relu'),                       # ✅
    layers.Dense(3,   activation='softmax')                     # ✅ softmax para 3 clases
])

---
---
# SECCIÓN 3 – Funciones de Pérdida (Loss)

---

## ¿Qué es una función de pérdida?

La función de pérdida (loss) mide qué tan equivocado está el modelo.
Durante el entrenamiento, el modelo **minimiza** esta función.
Si eliges la pérdida incorrecta, el modelo intentará minimizar algo que no tiene sentido para tu problema.

## Las pérdidas que necesitas saber

### `mse` – Mean Squared Error
```
MSE = promedio de (y_real - y_predicho)²
```
- Mide distancia numérica entre predicción y valor real.
- **Úsala en: REGRESIÓN.**
- ❌ Nunca en clasificación (las etiquetas 0/1 no son números continuos, son categorías).

### `binary_crossentropy`
- Mide qué tan lejos está una probabilidad predicha de la etiqueta real (0 o 1).
- **Úsala en: clasificación BINARIA** (sigmoid, 1 neurona de salida).

### `sparse_categorical_crossentropy`
- Como binary_crossentropy pero para N clases.
- **Úsala cuando las etiquetas son ENTEROS**: 0, 1, 2, 3... (lo más común).
- **Úsala en: clasificación MULTICLASE** con etiquetas como números enteros.

### `categorical_crossentropy`
- Igual a la anterior pero las etiquetas deben estar en formato **one-hot**: `[0,1,0,0]`.
- Si tienes etiquetas enteras, usa la `sparse_` versión.

## Tabla resumen

| Tarea | Formato de etiquetas | Loss |
|-------|---------------------|------|
| Regresión | Números continuos | `mse` |
| Binaria | 0 o 1 | `binary_crossentropy` |
| Multiclase | Enteros 0, 1, 2... | `sparse_categorical_crossentropy` |
| Multiclase | One-hot `[1,0,0]` | `categorical_crossentropy` |

### Ejercicio 3 – Pérdidas incorrectas

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso A: predecir el salario de una persona (número continuo)
model_A = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dense(1, activation='sigmoid')   # Error A1
])
model_A.compile(optimizer='adam', loss='binary_crossentropy')  # Error A2

# Caso B: clasificar el tipo de transacción (4 tipos), etiquetas son 0,1,2,3
model_B = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(15,)),
    layers.Dense(4, activation='softmax')
])
model_B.compile(optimizer='adam', loss='mse')   # Error B1

# Caso C: detectar si un mail es spam o no
model_C = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(200,)),
    layers.Dense(1, activation='relu')   # Error C1
])
model_C.compile(optimizer='adam', loss='categorical_crossentropy')  # Error C2

In [ ]:
# ✅ SOLUCIÓN

# CASO A – Regresión (salario)
# Error A1: sigmoid limita la salida a (0,1) → un salario puede ser 50.000.000
# Error A2: binary_crossentropy es para clasificación binaria
model_A = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dense(1)   # ✅ sin activación para regresión
])
model_A.compile(optimizer='adam', loss='mse')   # ✅

# CASO B – Clasificación multiclase (4 tipos, etiquetas enteras)
# Error B1: mse no tiene sentido para categorías
model_B = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(15,)),
    layers.Dense(4, activation='softmax')
])
model_B.compile(optimizer='adam', loss='sparse_categorical_crossentropy')   # ✅

# CASO C – Clasificación binaria (spam)
# Error C1: relu en salida → puede dar cualquier valor positivo, no una probabilidad
# Error C2: categorical_crossentropy es para multiclase con one-hot
model_C = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(200,)),
    layers.Dense(1, activation='sigmoid')   # ✅
])
model_C.compile(optimizer='adam', loss='binary_crossentropy')   # ✅

---
---
# SECCIÓN 4 – El Split de Datos y Data Leakage

---

## ¿Por qué dividimos los datos?

Queremos saber cómo se comportará el modelo con datos que **nunca ha visto**.
Si evaluamos en los mismos datos con que entrenamos, siempre va a parecer perfecto aunque no sirva en la vida real.

## Los tres conjuntos

```
Dataset completo
├── Train (60-70%)   ← el modelo aprende aquí
├── Validation (15%) ← ajustamos hiperparámetros (arquitectura, lr, etc.)
└── Test (15-20%)    ← evaluación FINAL, se toca solo UNA VEZ al final
```

## ¿Qué es Data Leakage?

**Data leakage** ocurre cuando información del test set contamina el proceso de entrenamiento.
El modelo parece bueno en evaluación pero falla en producción.

### La regla de oro
```
PRIMERO el split → LUEGO todo lo demás (fit, transform, SMOTE, etc.)
```

Cualquier objeto que aprenda algo de los datos (Scaler, Imputer, Tokenizer, Encoder, SMOTE)
debe hacer `.fit()` SOLO con datos de entrenamiento.

## Visualización del error más común

```python
# ❌ INCORRECTO – el scaler ve el test set
scaler.fit(X_completo)          # aprende media y std de TODOS los datos
X_train, X_test = split(...)   # el daño ya está hecho

# ✅ CORRECTO – el scaler solo ve el train
X_train, X_test = split(...)   # split primero
scaler.fit(X_train)            # aprende de train
scaler.transform(X_test)       # aplica lo aprendido al test
```

## `stratify`: qué es y cuándo usarlo

En clasificación con clases desbalanceadas, un split aleatorio puede poner todos los ejemplos
de la clase rara en el train y ninguno en el test (o viceversa).

`stratify=y` garantiza que la proporción de clases sea igual en train y test.

**Regla**: si la tarea es clasificación → siempre usa `stratify=y`.

### Ejercicio 4 – Data Leakage y split incorrecto

In [ ]:
# ❌ CÓDIGO CON ERRORES
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv('datos.csv')
X = df.drop('target', axis=1)
y = df['target']   # 95% clase 0, 5% clase 1

# Paso 1: imputar valores faltantes
imputer = SimpleImputer(strategy='median')
X_imp = imputer.fit_transform(X)         # ERROR 1

# Paso 2: escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)   # ERROR 1 (mismo)

# Paso 3: balancear con SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_scaled, y)   # ERROR 2

# Paso 4: split
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2)   # ERROR 3

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: imputer y scaler hacen .fit() en TODOS los datos antes del split
#   La mediana que aprende el imputer y la media/std del scaler incluyen
#   información del test set → el modelo ya 'ha visto' el test.
#
# ERROR 2: SMOTE se aplica a todo el dataset
#   SMOTE genera ejemplos SINTÉTICOS interpolando entre reales.
#   Al splitear después, esos sintéticos pueden quedar en el test set,
#   que ya no representa datos reales del mundo.
#
# ERROR 3: sin stratify con clases muy desbalanceadas (95/5)
#   El test podría quedar sin ningún ejemplo de clase 1.

# ✅ ORDEN CORRECTO

# 1. Split PRIMERO
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ✅ mantiene proporción 95/5
)

# 2. Fit de imputer y scaler SOLO en train
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)   # ✅ fit solo en train
X_test_imp  = imputer.transform(X_test)        # ✅ transform (sin fit) en test

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp) # ✅
X_test_sc  = scaler.transform(X_test_imp)      # ✅

# 3. SMOTE solo en train
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)  # ✅

# X_test_sc queda intacto con datos reales

---
---
# SECCIÓN 5 – Preprocesamiento

---

## Variables numéricas: ¿cuándo escalar?

Los modelos basados en distancias o gradientes son sensibles a las magnitudes.
Si 'ingresos' vale millones y 'edad' vale 18-80, las redes neuronales y el clustering
ignorarán edad y solo verán ingresos.

**Cuándo escalar:** siempre en redes neuronales (MLP, CNN, RNN), clustering (K-Means, DBSCAN), y KNN.
**No es estrictamente necesario en:** árboles de decisión, Random Forest.

| Escalador | Qué hace | Cuándo usarlo |
|-----------|----------|---------------|
| `StandardScaler` | Media 0, std 1 | Datos aproximadamente normales |
| `RobustScaler` | Usa mediana e IQR | Datos con outliers |
| `MinMaxScaler` | Escala a [0,1] | Cuando quieres rango fijo |

## Variables categóricas: ¿cómo encodear?

Las redes neuronales no entienden texto. Hay que convertir categorías a números.

| Tipo de variable | Ejemplo | Encoder |
|-----------------|---------|--------|
| **Nominal** (sin orden) | ciudad, color, marca | `OneHotEncoder` |
| **Ordinal** (con orden natural) | talla XS<S<M<L, nivel bajo<medio<alto | `OrdinalEncoder` |
| **Binaria** | sí/no, 1/0 | `.map({'SI':1, 'NO':0})` |

**Regla crítica:** usar `OrdinalEncoder` en una variable **nominal** introduce un orden falso
que el modelo interpreta como si Bogotá fuera mayor que Cali.

### Ejercicio 5 – Preprocesamiento incorrecto

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Dataset: edad (numérica), ciudad (nominal: Bogotá/Medellín/Cali/Barranquilla),
#          nivel_educacion (ordinal: basica<media<universitaria<postgrado),
#          ingresos (numérica con outliers)

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

# Error 1: OrdinalEncoder para variable NOMINAL (ciudad)
enc = OrdinalEncoder()
df['ciudad_enc'] = enc.fit_transform(df[['ciudad']])
# Resultado: Barranquilla=0, Bogotá=1, Cali=2, Medellín=3
# El modelo cree que Medellín > Cali > Bogotá → absurdo

# Error 2: OneHotEncoder para variable ORDINAL (nivel_educacion)
ohe = OneHotEncoder(sparse_output=False)
nivel_encoded = ohe.fit_transform(df[['nivel_educacion']])
# Pierde el orden: básica < media < universitaria < postgrado

# Error 3: StandardScaler en datos con muchos outliers
scaler = StandardScaler()
df['ingresos_sc'] = scaler.fit_transform(df[['ingresos']])
# Un outlier de 1 billón distorsiona la media y std enormemente

In [ ]:
# ✅ SOLUCIÓN
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ✅ Ciudad → OneHotEncoder (nominal, sin orden)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ciudad_enc = ohe.fit_transform(X_train[['ciudad']])

# ✅ Nivel educación → OrdinalEncoder CON el orden definido explícitamente
orden_educacion = [['basica', 'media', 'universitaria', 'postgrado']]
oe = OrdinalEncoder(categories=orden_educacion)
nivel_enc = oe.fit_transform(X_train[['nivel_educacion']])
# Resultado: basica=0, media=1, universitaria=2, postgrado=3  ✅ orden correcto

# ✅ Ingresos con outliers → RobustScaler
rs = RobustScaler()
ingresos_sc = rs.fit_transform(X_train[['ingresos']])

# Forma organizada con ColumnTransformer:
preprocessor = ColumnTransformer([
    ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['ciudad']),
    ('ord', OrdinalEncoder(categories=orden_educacion),                  ['nivel_educacion']),
    ('rob', RobustScaler(),                                              ['ingresos', 'edad'])
])

---
---
# SECCIÓN 6 – MLP: Redes Neuronales Densas

---

## ¿Qué es un MLP (Multi-Layer Perceptron)?

Un MLP es una red neuronal formada por capas de neuronas "densas" (Dense),
donde cada neurona está conectada con todas las de la capa siguiente.

```
[entrada] → [capa oculta 1] → [capa oculta 2] → [salida]
```

Cada capa aprende representaciones cada vez más abstractas de los datos.

## Anatomía de un modelo en Keras

```python
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(n_features,)),  # 1ª capa: define input
    layers.Dense(64,  activation='relu'),   # capas ocultas: relu
    layers.Dense(1,   activation='sigmoid') # capa de salida: depende del problema
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, validation_split=0.2)
```

## Overfitting y cómo combatirlo

**Overfitting** = el modelo memoriza el conjunto de entrenamiento pero no generaliza.
Se ve cuando: `train_accuracy >> val_accuracy`.

| Técnica | Qué hace |
|---------|----------|
| `Dropout(p)` | Apaga aleatoriamente el p% de neuronas en cada paso de entrenamiento |
| `l2` regularización | Penaliza pesos muy grandes |
| `EarlyStopping` | Para el entrenamiento si la val_loss deja de mejorar |
| Menos capas/neuronas | Modelo más pequeño → menos capacidad de memorizar |

## Learning rate

Controla qué tan grandes son los pasos que da el modelo al actualizar los pesos.

| Valor lr | Efecto |
|----------|--------|
| Demasiado alto (ej: 10) | La loss explota a `NaN` |
| Alto (ej: 0.1) | La loss oscila, no converge bien |
| **Bueno (0.001)** | Converge de forma estable ← **default de Adam** |
| Bajo (0.00001) | Converge muy despacio, necesita muchas épocas |

### Ejercicio 6 – MLP completo con múltiples errores

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: predecir si un paciente tiene diabetes (binario: sí/no)
# 20 features numéricos, 1000 ejemplos

from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Dense(4096, activation='relu', input_shape=(20,)),  # Error 1
    layers.Dense(4096, activation='relu'),                     # Error 1
    layers.Dense(4096, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5.0),    # Error 2
    loss='binary_crossentropy',
    metrics=['accuracy']
)
# Se entrena sin validación ni EarlyStopping
model.fit(X_train, y_train, epochs=500)                    # Error 3

# Evaluación
y_pred = model.predict(X_test)                             # Error 4
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))                      # Error 4

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: modelo gigante (4096 neuronas × 3 capas) para solo 1000 ejemplos
#   Con más parámetros que datos, el modelo memoriza en vez de aprender.
#   CORRECCIÓN: usar una arquitectura proporcional al tamaño del dataset.
#
# ERROR 2: learning_rate=5.0
#   Produce gradientes explosivos → loss = NaN desde las primeras épocas.
#   CORRECCIÓN: lr=0.001 (default de Adam).
#
# ERROR 3: 500 épocas sin validación ni EarlyStopping
#   El modelo sobreajusta inevitablemente. No sabemos cuándo parar.
#   CORRECCIÓN: validation_split + EarlyStopping.
#
# ERROR 4: model.predict() devuelve probabilidades (shape N×1), no clases (shape N,)
#   accuracy_score necesita enteros 0 o 1, no floats como 0.73.
#   CORRECCIÓN: convertir con umbral 0.5.

import numpy as np
from tensorflow.keras import callbacks

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dropout(0.3),                    # ✅ regularización
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),  # ✅
    loss='binary_crossentropy',
    metrics=['accuracy']
)

es = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
model.fit(
    X_train, y_train,
    epochs=500,
    validation_split=0.2,   # ✅ separa 20% para validar
    callbacks=[es]          # ✅ para solo cuando no mejora
)

# ✅ Evaluación correcta
y_prob = model.predict(X_test).flatten()    # probabilidades: shape (N,)
y_pred = (y_prob > 0.5).astype(int)        # clases: 0 o 1

from sklearn.metrics import accuracy_score, classification_report
print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

---
---
# SECCIÓN 7 – CNN: Redes Convolucionales para Imágenes

---

## ¿Por qué no usar un MLP para imágenes?

Una imagen de 32×32×3 (CIFAR-10) tiene 3.072 valores.
Si la aplanamos y usamos un MLP, perdemos la información **espacial** (qué píxel está al lado de cuál).
Una CNN preserva esa información usando **filtros** que recorren la imagen.

## Componentes de una CNN

### `Conv2D(filtros, (k,k), activation='relu')`
Aplica un filtro de tamaño k×k sobre la imagen y detecta patrones locales (bordes, texturas, formas).
- Con 32 filtros → produce 32 mapas de características.
- `padding='same'` → mantiene el tamaño H×W.
- `padding='valid'` (default) → reduce H y W en cada capa.

### `MaxPooling2D((2,2))`
Reduce el tamaño a la mitad tomando el máximo en cada ventana 2×2.
Hace la red más pequeña y resistente a pequeños desplazamientos.

### `Flatten()`
Convierte el tensor 3D (altura × ancho × canales) en un vector 1D para pasarlo a capas Dense.
**Esta capa es obligatoria** entre la parte convolucional y la parte Dense.

## Reglas críticas de CNN

1. **Normalizar píxeles** antes de entrenar: `imagen / 255.0` → valores en [0, 1].
2. **input_shape** siempre debe incluir el canal: `(H, W, C)`.
   - RGB: `(32, 32, 3)` — 3 canales de color.
   - Escala de grises: `(28, 28, 1)` — 1 canal.
3. **Flatten** va SIEMPRE antes de Dense.
4. Si hay imágenes en escala de grises, el array debe tener 4 dimensiones: `(N, H, W, 1)`. Usar `.reshape(-1, H, W, 1)`.

## Arquitectura tipo para imágenes

```
Imagen (32,32,3)
  ↓
Conv2D(32, 3×3, relu, padding='same')
  ↓
MaxPooling2D(2×2)        → (16,16,32)
  ↓
Conv2D(64, 3×3, relu, padding='same')
  ↓
MaxPooling2D(2×2)        → (8,8,64)
  ↓
Flatten()                → (4096,)
  ↓
Dense(128, relu)
  ↓
Dense(10, softmax)       → predicción de 10 clases
```

### Ejercicio 7.A – CNN con errores estructurales

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: clasificar imágenes CIFAR-10 (32×32 píxeles, color RGB, 10 clases)
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
# train_images.shape = (50000, 32, 32, 3), valores en [0, 255]

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2,2)),
    layers.Dense(64, activation='relu'),          # Error 1
    layers.Dense(10, activation='relu')           # Error 2
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',                   # Error 3
    metrics=['accuracy']
)
model.fit(train_images, train_labels, epochs=5)  # Error 4: imágenes sin normalizar

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: Dense sin Flatten antes
#   Después de MaxPooling el tensor es 3D: (16, 16, 32).
#   Dense solo puede procesar vectores 1D.
#   Necesitas Flatten() en el medio.
#
# ERROR 2: relu en salida para 10 clases
#   relu puede dar valores > 1 y no genera distribución de probabilidad.
#   Para 10 clases → softmax.
#
# ERROR 3: binary_crossentropy para 10 clases
#   binary_crossentropy es para 2 clases (salida: 1 neurona con sigmoid).
#   Para 10 clases con etiquetas enteras → sparse_categorical_crossentropy.
#
# ERROR 4: imágenes en rango [0, 255] sin normalizar
#   Valores tan grandes saturan las activaciones y hacen inestable el entrenamiento.
#   Dividir entre 255.0 para llevar al rango [0, 1].

train_images = train_images / 255.0    # ✅ normalizar
test_images  = test_images  / 255.0

model = models.Sequential([
    layers.Conv2D(32, (3,3), padding='same', activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),                              # ✅ Flatten antes de Dense
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')         # ✅ softmax para 10 clases
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',        # ✅
    metrics=['accuracy']
)
model.fit(train_images, train_labels, epochs=10, validation_split=0.1)

### Ejercicio 7.B – CNN para imágenes en escala de grises (MNIST)

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: clasificar dígitos escritos a mano (MNIST, 28×28, escala de grises, 10 clases)
from tensorflow import keras

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
# X_train.shape = (60000, 28, 28)  ← 3D, sin canal de color

X_train = X_train / 255.0
X_test  = X_test  / 255.0

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28, 28)),  # Error 1
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])
# El modelo va a fallar con error de shape

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: input_shape=(28, 28) – faltan las dimensiones del canal
#   Conv2D espera tensores 4D: (batch, height, width, channels).
#   Para escala de grises: channels=1 → input_shape=(28, 28, 1).
#   También hay que hacer reshape al array: (60000, 28, 28) → (60000, 28, 28, 1).

import numpy as np

(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

X_train = X_train.reshape(-1, 28, 28, 1) / 255.0   # ✅ añadir canal + normalizar
X_test  = X_test.reshape(-1, 28, 28, 1)  / 255.0

model = models.Sequential([
    layers.Conv2D(32, (3,3), padding='same', activation='relu',
                  input_shape=(28, 28, 1)),           # ✅ con canal
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), padding='same', activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

---
---
# SECCIÓN 8 – Embeddings y NLP

---

## ¿Por qué no podemos meter texto directamente a una red?

Las redes neuronales solo procesan **números**. El texto es texto.
Necesitamos convertirlo en números de una manera que preserve el significado.

## El pipeline completo de NLP

```
Texto crudo
     ↓
Tokenización        →  cada palabra se convierte en un número entero (índice)
     ↓
Padding             →  todas las secuencias tienen la misma longitud
     ↓
Embedding Layer     →  cada índice se convierte en un vector denso aprendible
     ↓
Red (LSTM / Dense)  →  procesa los vectores
     ↓
Salida
```

## ¿Qué es un Embedding?

Un embedding convierte un índice entero (número de palabra) en un **vector denso de dimensión fija**.
Por ejemplo, con `output_dim=64`:

```
"banco" → índice 147 → vector [0.23, -0.51, 0.88, ..., 0.14]  (64 valores)
```

El modelo **aprende** estos vectores durante el entrenamiento.
Palabras con significado similar terminan con vectores similares.

## One-hot vs Embedding

| | One-Hot | Embedding |
|-|---------|----------|
| Dimensión | vocab_size (ej: 50.000) | embed_dim (ej: 64) |
| Tipo | Disperso (casi todo 0) | Denso |
| Semántica | Ninguna (cada palabra es ortogonal) | Capturada |
| Para texto grande | Inviable (RAM) | Eficiente |

## La capa `Embedding` en Keras

```python
layers.Embedding(
    input_dim=VOCAB_SIZE,    # número de palabras distintas en el vocabulario
    output_dim=EMBED_DIM,    # tamaño del vector de embedding (ej: 64, 128, 300)
    input_length=MAX_LEN     # longitud de las secuencias (después de padding)
)
```

**Importante**: el Tokenizer y el Embedding deben usar el mismo `VOCAB_SIZE`.

### Ejercicio 8 – Pipeline de NLP con errores

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: clasificar tickets de soporte en 5 categorías usando texto en español
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

df = pd.read_csv('tickets.csv')
X = df['texto']
y = df['categoria']

# Error 1: tokenizer hace fit en TODOS los textos (leakage)
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X)                         # Error 1
sequences = tokenizer.texts_to_sequences(X)
X_padded  = pad_sequences(sequences, maxlen=100)

X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2)

VOCAB_SIZE = 5000
MAX_LEN    = 100

model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(MAX_LEN,)),  # Error 2
    layers.Dense(5, activation='sigmoid')                          # Error 3
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',                                    # Error 4
    metrics=['accuracy']
)

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: tokenizer.fit en todos los textos antes del split
#   El vocabulario se construye viendo el test set → leakage.
#   CORRECCIÓN: split primero, fit del tokenizer solo en textos de train.
#
# ERROR 2: Dense directamente sobre los índices de tokens
#   Los índices (1, 42, 987...) son números arbitrarios, no vectores con significado.
#   Se necesita una capa Embedding que los convierta en vectores densos.
#
# ERROR 3: sigmoid en salida para 5 clases
#   sigmoid produce probabilidades independientes (no suman 1).
#   Para 5 clases → softmax.
#
# ERROR 4: binary_crossentropy para 5 clases
#   CORRECCIÓN: sparse_categorical_crossentropy.

from sklearn.preprocessing import LabelEncoder

# ✅ Split PRIMERO
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['texto'].tolist(), y, test_size=0.2, random_state=42, stratify=y
)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test  = le.transform(y_test)

# ✅ Tokenizer fit solo en train
VOCAB_SIZE = 5000
MAX_LEN    = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_raw)           # ✅ solo train

X_train_seq = tokenizer.texts_to_sequences(X_train_raw)
X_test_seq  = tokenizer.texts_to_sequences(X_test_raw)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='pre')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='pre')

# ✅ Modelo con Embedding
model = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, 64, input_length=MAX_LEN),   # ✅ Embedding
    layers.GlobalAveragePooling1D(),                           # promedia los tokens
    layers.Dense(64, activation='relu'),
    layers.Dense(5, activation='softmax')                      # ✅ softmax para 5 clases
])
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',                    # ✅
    metrics=['accuracy']
)

---
---
# SECCIÓN 9 – RNN, LSTM y GRU

---

## ¿Para qué sirven las redes recurrentes?

Un MLP procesa cada ejemplo de forma independiente, ignorando el orden.
Para texto y series de tiempo, el **orden importa**: "el banco aprobó el préstamo" vs "el préstamo aprobó el banco".

Las redes recurrentes procesan una secuencia paso a paso, manteniendo una **memoria** de lo anterior.

## SimpleRNN vs LSTM vs GRU

| Modelo | Memoria | Cuándo usarlo |
|--------|---------|---------------|
| `SimpleRNN` | Corta (olvidadiza) | Secuencias muy cortas (< 30 pasos) |
| `LSTM` | Larga (compuerta de olvido) | Texto largo, dependencias distantes |
| `GRU` | Larga (más simple que LSTM) | Como LSTM pero más rápido, menos parámetros |

**Problema del SimpleRNN**: en secuencias largas sufre de **vanishing gradient**.
El gradiente se hace tan pequeño que la red "olvida" lo que leyó al principio.
LSTM y GRU lo resuelven con mecanismos de compuertas.

## `return_sequences`: la regla que más confunde

```
return_sequences=False (default):
  Entrada: (batch, timesteps, features)
  Salida:  (batch, units)          ← solo el ÚLTIMO estado
  ✅ Úsalo si el siguiente es Dense

return_sequences=True:
  Entrada: (batch, timesteps, features)
  Salida:  (batch, timesteps, units) ← un estado por cada paso
  ✅ Úsalo si el siguiente es OTRO LSTM/GRU
```

## Bidireccional: cuándo usarlo y cuándo no

Un `Bidirectional(LSTM)` lee la secuencia en ambos sentidos (→ y ←).
Tiene acceso al **contexto futuro**, lo cual es bueno para clasificación de texto
pero **inválido** para generación de texto o predicción de series de tiempo.

| Tarea | Bidireccional |
|-------|----------|
| Clasificación de sentimientos | ✅ sí (tiene toda la oración) |
| Detección de entidades (NER) | ✅ sí |
| Generación de texto | ❌ no (ve el futuro, hace trampa) |
| Predicción de series de tiempo | ❌ no (el futuro no existe aún) |

### Ejercicio 9.A – LSTM apilados con return_sequences incorrecto

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: análisis de sentimientos con dos capas LSTM
model = keras.Sequential([
    layers.Embedding(10000, 64, input_length=200),
    layers.LSTM(64),                              # Error 1
    layers.LSTM(32),                              # recibe tensor 2D, necesita 3D
    layers.Dense(1, activation='sigmoid')
])
# Error: Input 0 of layer lstm_1 is incompatible with the layer:
#        expected ndim=3, found ndim=2

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: el primer LSTM tiene return_sequences=False (default)
#   Devuelve shape (batch, 64): tensor 2D.
#   El segundo LSTM espera shape (batch, timesteps, features): tensor 3D.
#
# REGLA: si hay N capas LSTM apiladas, las primeras N-1 deben tener
#         return_sequences=True. Solo la última puede tener False.

model = keras.Sequential([
    layers.Embedding(10000, 64, input_length=200),
    layers.LSTM(64, return_sequences=True),   # ✅ devuelve secuencia para el siguiente LSTM
    layers.LSTM(32, return_sequences=False),  # ✅ último LSTM: devuelve solo el estado final
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

### Ejercicio 9.B – LSTM para series de tiempo sin reshape

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: predecir el precio de una acción usando los últimos 30 días
import numpy as np

precios = np.array([...])  # array de 500 días

# Crear ventanas de 30 días
X = np.array([precios[i:i+30] for i in range(len(precios)-30)])
y = np.array([precios[i+30]   for i in range(len(precios)-30)])
print(X.shape)   # (470, 30)  ← 2D

model = keras.Sequential([
    layers.LSTM(64, input_shape=(30,)),   # Error 1: LSTM espera 3D
    layers.Dense(1)
])

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: LSTM espera (batch, timesteps, features)
#   Con una sola variable (precio), features=1.
#   X tiene shape (470, 30) → hay que hacerlo (470, 30, 1).

X = np.array([precios[i:i+30] for i in range(len(precios)-30)])
X = X.reshape(X.shape[0], X.shape[1], 1)   # ✅ (470, 30, 1)

y = np.array([precios[i+30] for i in range(len(precios)-30)])

model = keras.Sequential([
    layers.LSTM(64, input_shape=(30, 1)),   # ✅ (timesteps=30, features=1)
    layers.Dense(1)                          # regresión: sin activación
])
model.compile(optimizer='adam', loss='mse')   # ✅ regresión → mse

---
---
# SECCIÓN 10 – Clustering

---

## ¿Qué es el clustering?

El clustering es aprendizaje **no supervisado**: no tenemos etiquetas.
El objetivo es encontrar **grupos naturales** en los datos.

## K-Means

Divide los datos en K grupos (clusters) esféricos minimizando la **inercia**
(suma de distancias al centroide del grupo).

**Limitaciones:**
- Solo funciona bien con grupos de forma esférica.
- Sensible a outliers (un punto muy alejado arrastra el centroide).
- Hay que definir K de antemano.
- **Requiere que los datos estén escalados**: K-Means usa distancia Euclidiana.
  Si una variable tiene valores en millones y otra en decenas, la primera domina completamente.

**Cómo elegir K:**
1. **Método del codo**: graficar inercia vs K, buscar el punto donde la curva "dobla".
2. **Silhouette score**: valor entre -1 y 1. Más alto = clusters más separados y compactos.

## DBSCAN

Encuentra grupos de forma arbitraria y marca puntos aislados como **ruido** (label = -1).

**Parámetros:**
- `eps`: radio de vecindad. Puntos dentro de esta distancia son vecinos.
- `min_samples`: mínimo de puntos en la vecindad para ser "core point".

**Ventajas sobre K-Means:**
- No hay que definir K.
- Detecta outliers automáticamente.
- Funciona con grupos de forma arbitraria.

## Clustering jerárquico

Construye una jerarquía de clusters que se puede visualizar en un **dendrograma**.
Se puede cortar en cualquier nivel para obtener el número de clusters deseado.

## Reglas críticas de clustering

1. **Siempre escalar** antes de K-Means y DBSCAN.
2. **Solo variables numéricas**: K-Means no puede procesar texto o categorías directas.
3. **Nunca usar accuracy** para evaluar clustering (no hay etiquetas reales).
   Usar: silhouette score, inercia (para K-Means), o métricas externas (ARI, NMI) si tienes etiquetas de referencia.
4. Los **labels de K-Means son arbitrarios**: el cluster 0 hoy puede ser el cluster 2 mañana.

### Ejercicio 10 – K-Means con múltiples errores

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: segmentar clientes de una aerolínea
# Variables: edad, ingresos, vuelos_año, ciudad_origen (texto), puntos_fidelidad

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

df = pd.read_csv('clientes.csv')

# Error 1: incluir variable de texto directamente
X = df[['edad', 'ingresos', 'vuelos_año', 'ciudad_origen', 'puntos_fidelidad']]

# Error 2: k elegido sin justificación
km = KMeans(n_clusters=7)  # ¿por qué 7?

# Error 3: datos sin escalar
km.fit(X)
labels = km.labels_

# Error 4: evaluar con accuracy (no tiene sentido en clustering)
print(accuracy_score(df['segmento_real'], labels))

In [ ]:
# ✅ SOLUCIÓN
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import silhouette_score

df = pd.read_csv('clientes.csv')

# ✅ ERROR 1: solo variables numéricas
X = df[['edad', 'ingresos', 'vuelos_año', 'puntos_fidelidad']]

# ✅ ERROR 3: escalar ANTES del clustering
scaler = RobustScaler()               # RobustScaler es resistente a outliers
X_scaled = scaler.fit_transform(X)

# ✅ ERROR 2: elegir k con método del codo + silhouette
inertias = []
silhouettes = []
k_range = range(2, 10)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(k_range, inertias, 'o-'); ax1.set_xlabel('k'); ax1.set_ylabel('Inercia')
ax2.plot(k_range, silhouettes, 'o-', color='orange')
ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette')
plt.show()

# Con k elegido:
k_optimo = 4   # basado en las gráficas
km_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
labels = km_final.fit_predict(X_scaled)

# ✅ ERROR 4: métricas correctas para clustering
score = silhouette_score(X_scaled, labels)
print(f'Silhouette score: {score:.3f}')   # entre -1 y 1, más alto es mejor
print(f'Inercia: {km_final.inertia_:.1f}')

---
---
# SECCIÓN 11 – Naive Bayes

---

## ¿Qué es Naive Bayes?

Naive Bayes es un clasificador basado en el **Teorema de Bayes**:

```
P(clase | datos) = P(datos | clase) × P(clase) / P(datos)
```

Para clasificar, elegimos la clase que maximiza `P(clase | datos)`.

El supuesto **"Naive"** (ingenuo) es que todas las características son **independientes entre sí**
dado el valor de la clase. Esto raramente es verdad, pero el modelo funciona bien en la práctica.

## Generativo vs Discriminativo

Esta es una pregunta conceptual muy probable en el parcial.

| | Generativo | Discriminativo |
|-|------------|----------------|
| **Aprende** | P(X, Y) = P(X\|Y) × P(Y) | P(Y\|X) directamente |
| **Puede hacer** | Clasificar Y dado X, y también **generar** nuevos X dado Y | Solo clasificar |
| **Ejemplos** | Naive Bayes, GAN, VAE | Regresión logística, SVM, Redes neuronales |
| **Naive Bayes** | ✅ Es generativo | — |

**Por qué Naive Bayes es generativo:** aprende la distribución de los datos
para cada clase (media y varianza de cada feature dado Y=clase).
Con eso puede generar nuevas muestras como se vio en la práctica: `np.random.normal(mean, std)`.

## Tipos de Naive Bayes

| Tipo | Para qué datos |
|------|----------------|
| `GaussianNB` | Features **continuas** (supone distribución Normal) |
| `MultinomialNB` | **Conteos** de palabras, frecuencias enteras |
| `BernoulliNB` | Features **binarias** (presencia/ausencia) |

## Evaluación correcta

Si el dataset está desbalanceado (ej: 98% no-fraude, 2% fraude),
un modelo que siempre predice "no fraude" tiene **98% accuracy** pero es inútil.
Usar: `classification_report`, `ROC-AUC`, `confusion_matrix`.

### Ejercicio 11 – Naive Bayes: tipo incorrecto y métrica incorrecta

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso: clasificar correos como spam/no spam usando frecuencia de palabras
# X tiene conteos de palabras: [[0, 2, 0, 5, 1, ...], ...]  (enteros >= 0)

from sklearn.naive_bayes import GaussianNB   # Error 1: tipo incorrecto
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# El dataset tiene 97% no-spam, 3% spam
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)  # Error 2: sin stratify

bayes = GaussianNB()
bayes.fit(X_train, y_train)
y_pred = bayes.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')  # Error 3: métrica incorrecta
# Imprime 0.97 ← parece perfecto pero el modelo puede no detectar ningún spam

In [ ]:
# ✅ SOLUCIÓN
#
# ERROR 1: GaussianNB para conteos de palabras
#   GaussianNB asume que los features siguen una distribución Normal continua.
#   Los conteos de palabras son enteros no negativos → MultinomialNB.
#
# ERROR 2: sin stratify con dataset desbalanceado (97/3)
#   El test set podría no tener ningún spam → no se puede evaluar bien.
#
# ERROR 3: accuracy en dataset desbalanceado
#   0.97 accuracy puede significar que el modelo predice siempre "no spam".
#   Para saber si detecta spam, mirar el recall de la clase spam.

from sklearn.naive_bayes import MultinomialNB   # ✅ para conteos
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
import seaborn as sns
import matplotlib.pyplot as plt

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # ✅
)

bayes = MultinomialNB(alpha=1.0)   # alpha: suavizado de Laplace (evita prob=0)
bayes.fit(X_train, y_train)
y_pred = bayes.predict(X_test)
y_prob = bayes.predict_proba(X_test)[:, 1]

# ✅ Métricas completas
print(classification_report(y_test, y_pred, target_names=['No spam', 'Spam']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No spam','Spam'], yticklabels=['No spam','Spam'])
plt.xlabel('Predicho'); plt.ylabel('Real'); plt.show()

# Interpretar: ¿cuántos spams reales detectó? → fila 'Spam', columna 'Spam' (TP)
# ¿Cuántos spams se escaparon? → fila 'Spam', columna 'No spam' (FN)

---
---
# SECCIÓN 12 – Métricas de Evaluación

---

## ¿Por qué no basta con la accuracy?

La **accuracy** (exactitud) mide el porcentaje de predicciones correctas.
Parece perfecta, pero falla en cuanto el dataset tiene clases desbalanceadas.

**Ejemplo clásico:** 98% de las transacciones son normales, 2% son fraude.
Un modelo que predice SIEMPRE "no es fraude" tiene **98% accuracy**.
Pero ese modelo no detecta ni un solo fraude: es completamente inútil.

## La matriz de confusión

Para clasificación binaria:

```
                 Predicho: NEG    Predicho: POS
  Real: NEG  [[    TN (acierto),    FP (falsa alarma)  ]]
  Real: POS  [[    FN (fallo),      TP (detección)     ]]
```

- **TN** (True Negative):  negativos correctamente predichos como negativos.
- **FP** (False Positive): negativos incorrectamente predichos como positivos → **alarma falsa**.
- **FN** (False Negative): positivos incorrectamente predichos como negativos → **el fallo más peligroso** en medicina/fraude.
- **TP** (True Positive):  positivos correctamente predichos como positivos.

## Precision, Recall y F1

```
Precision = TP / (TP + FP)  →  de todos los que predije como positivos, ¿cuántos lo eran?
Recall    = TP / (TP + FN)  →  de todos los positivos reales, ¿cuántos detecté?
F1        = 2 × Precision × Recall / (Precision + Recall)  →  media armónica entre ambos
```

| Situación | Métrica prioritaria |
|-----------|-------------------|
| Medicina (no quiero perder enfermos) | **Recall** |
| Spam (no quiero borrar correos importantes) | **Precision** |
| Balance entre ambos | **F1** |
| Dataset desbalanceado general | **ROC-AUC** |

## Métricas por tipo de problema

| Tipo | Métricas correctas | Métricas incorrectas |
|------|-------------------|--------------------|
| Clasificación balanceada | Accuracy, F1 | — |
| Clasificación desbalanceada | F1 (macro), ROC-AUC, Recall de clase positiva | Accuracy |
| Regresión | MSE, RMSE, MAE, R² | Accuracy, F1 |
| Clustering | Silhouette, Inercia | Accuracy |

## F1 macro vs F1 weighted

- **F1 macro**: promedio simple de F1 por clase → todas las clases pesan igual.
- **F1 weighted**: promedio ponderado por tamaño de clase → las clases grandes pesan más.

En datasets desbalanceados, `weighted` puede dar 0.99 aunque la clase rara tenga F1=0.
Para detectar si fallas en clases minoritarias → usar **F1 macro** o ver el classification_report completo.

### Ejercicio 12 – Métricas incorrectas en múltiples situaciones

In [ ]:
# ❌ CÓDIGO CON ERRORES
# Caso A: detectar fraude (1% fraudes en el dataset)
from sklearn.metrics import accuracy_score, r2_score, f1_score

# Modelo de clustering K-Means evaluado con accuracy
labels = km.fit_predict(X)  # Error A
print(accuracy_score(y_real, labels))

# Modelo de detección de fraude evaluado con accuracy
y_pred_fraude = clf.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred_fraude):.2f}')  # Error B: 0.99 pero es inútil

# Modelo de clasificación multiclase (5 clases, muy desbalanceadas)
# Clase 1: 1000 muestras, Clase 2-5: 10 muestras cada una
f1_w = f1_score(y_test, y_pred, average='weighted')  # Error C: oculta clases raras
print(f'F1 Weighted: {f1_w:.3f}')  # puede dar 0.99 aunque falle en clases 2-5

# Modelo de regresión evaluado con accuracy
y_pred_reg = modelo_reg.predict(X_test)
print(r2_score(y_test, y_pred_fraude))  # Error D: mezclando variables

In [ ]:
# ✅ SOLUCIÓN
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    f1_score, silhouette_score, mean_squared_error, mean_absolute_error, r2_score
)
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ✅ CASO A: Clustering → silhouette (no accuracy)
score = silhouette_score(X_scaled, labels)
print(f'Silhouette: {score:.3f}  (1.0=perfecto, 0=clusters solapados, -1=mal clustering)')

# ✅ CASO B: Fraude → ROC-AUC + classification_report
y_prob_fraude = clf.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred_fraude, target_names=['Normal', 'Fraude']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_fraude):.4f}')
# Ver el Recall de 'Fraude': ¿detectamos suficientes fraudes reales?

# ✅ CASO C: Multiclase desbalanceada → F1 macro
f1_macro = f1_score(y_test, y_pred, average='macro')       # ✅ pesan igual todas
f1_wght  = f1_score(y_test, y_pred, average='weighted')    # engañoso con desbalance
print(f'F1 Macro:    {f1_macro:.3f}')  # refleja si fallas en clases raras
print(f'F1 Weighted: {f1_wght:.3f}')  # puede estar inflado
print(classification_report(y_test, y_pred))               # siempre mirar clase por clase

# ✅ CASO D: Regresión → MSE, MAE, R² (no accuracy)
mse = mean_squared_error(y_test, y_pred_reg)
mae = mean_absolute_error(y_test, y_pred_reg)
r2  = r2_score(y_test, y_pred_reg)
print(f'MSE: {mse:.2f}  MAE: {mae:.2f}  R²: {r2:.4f}')
# R²: 1.0 = perfecto, 0 = tan malo como predecir la media, negativo = peor que la media

---
---
# SECCIÓN 13 – Caso Completo Final

---

## Lee el caso, detecta TODOS los errores, luego mira la solución

**Caso:** Un banco quiere predecir si un cliente pedirá soporte en los próximos 30 días
basándose en su comportamiento. El dataset tiene:
- `meses_cliente` (numérica)
- `transacciones_mes` (numérica con outliers)
- `tipo_plan` (ordinal: básico < estándar < premium)
- `ciudad` (nominal: 5 ciudades)
- `ultima_queja` (texto libre)
- `pedira_soporte` (0/1, 85% son 0)

In [ ]:
# ❌ CÓDIGO CON ERRORES – Encuentra todos
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score

df = pd.read_csv('banco.csv')

# ── Preprocesamiento ──
imputer = SimpleImputer(strategy='mean')
df[['meses_cliente', 'transacciones_mes']] = imputer.fit_transform(
    df[['meses_cliente', 'transacciones_mes']]
)                                               # Error 1

scaler = StandardScaler()
df[['meses_cliente', 'transacciones_mes']] = scaler.fit_transform(
    df[['meses_cliente', 'transacciones_mes']]
)                                               # Error 2

enc = OrdinalEncoder()
df['ciudad_enc'] = enc.fit_transform(df[['ciudad']])  # Error 3

# Ignoramos 'ultima_queja' directamente

X = df[['meses_cliente', 'transacciones_mes', 'tipo_plan', 'ciudad_enc']]
y = df['pedira_soporte']

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)         # Error 4

X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2
)                                               # Error 5

# ── Modelo ──
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(512, activation='relu'),
    layers.Dense(512, activation='relu'),
    layers.Dense(2, activation='softmax')        # Error 6
])
model.compile(
    optimizer=keras.optimizers.Adam(0.5),        # Error 7
    loss='categorical_crossentropy',             # Error 8
    metrics=['accuracy']
)
model.fit(X_train, y_train, epochs=200)          # Error 9

y_pred = np.argmax(model.predict(X_test), axis=1)
print(accuracy_score(y_test, y_pred))            # Error 10

In [ ]:
# ✅ SOLUCIÓN COMPLETA – 10 errores
#
# ERROR 1: imputer.fit_transform() en todo el dataset antes del split
#   Data leakage: la mediana del test contaminó la imputación.
#   CORRECCIÓN: split primero, imputer.fit() solo en train.
#
# ERROR 2: StandardScaler antes del split
#   Data leakage: la media y std del test contaminaron el escalado.
#   Además: transacciones_mes tiene outliers → RobustScaler es mejor.
#   CORRECCIÓN: split primero, scaler.fit() solo en train.
#
# ERROR 3: OrdinalEncoder para 'ciudad' (variable NOMINAL)
#   Ciudad no tiene orden → introduce relaciones falsas.
#   CORRECCIÓN: OneHotEncoder.
#
# ERROR 4: SMOTE antes del split
#   Los ejemplos sintéticos pueden quedar en el test set.
#   CORRECCIÓN: split primero, SMOTE solo en X_train.
#
# ERROR 5: sin stratify con dataset desbalanceado (85/15)
#   CORRECCIÓN: stratify=y.
#
# ERROR 6: Dense(2, softmax) para clasificación binaria
#   Técnicamente funciona pero la convención es Dense(1, sigmoid) para binario.
#   Con 2 neuronas y softmax: la pérdida debe ser categorical_crossentropy
#   y las etiquetas deben ser one-hot. Más complejo sin beneficio.
#   CORRECCIÓN: Dense(1, sigmoid).
#
# ERROR 7: learning_rate=0.5
#   Demasiado alto para Adam. Causará inestabilidad o divergencia.
#   CORRECCIÓN: lr=0.001.
#
# ERROR 8: categorical_crossentropy con etiquetas binarias (0/1)
#   categorical_crossentropy espera etiquetas one-hot [1,0] / [0,1].
#   CORRECCIÓN: binary_crossentropy.
#
# ERROR 9: 200 épocas sin validation ni EarlyStopping
#   El modelo sobreajusta. No sabemos cuándo es el mejor punto.
#   CORRECCIÓN: validation_split + EarlyStopping.
#
# ERROR 10: accuracy con dataset desbalanceado (85/15)
#   Un modelo que predice siempre 'no pide soporte' tiene 85% accuracy.
#   CORRECCIÓN: classification_report + ROC-AUC.

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv('banco.csv')
X = df[['meses_cliente', 'transacciones_mes', 'tipo_plan', 'ciudad']]
y = df['pedira_soporte']

# ✅ 1. Split PRIMERO con stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ✅ 2. Preprocesamiento correcto por tipo de columna
num_rob = Pipeline([                                  # con outliers → RobustScaler
    ('imp', SimpleImputer(strategy='median')),
    ('sc', RobustScaler())
])
num_std = Pipeline([                                  # sin outliers → StandardScaler
    ('imp', SimpleImputer(strategy='median')),
    ('sc', StandardScaler())
])
ord_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(categories=[['básico','estándar','premium']]))
])                                                    # ordinal con orden definido
nom_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])                                                    # ✅ nominal → OneHot

prep = ColumnTransformer([
    ('rob',  num_rob,  ['transacciones_mes']),
    ('std',  num_std,  ['meses_cliente']),
    ('ord',  ord_pipe, ['tipo_plan']),
    ('nom',  nom_pipe, ['ciudad'])
])

X_train_prep = prep.fit_transform(X_train)   # ✅ fit solo en train
X_test_prep  = prep.transform(X_test)        # ✅ transform en test

# ✅ 3. SMOTE solo en train
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_prep, y_train)

# ✅ 4. Modelo correcto
n_features = X_train_res.shape[1]
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(n_features,)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')      # ✅ sigmoid para binario
])
model.compile(
    optimizer=keras.optimizers.Adam(0.001),    # ✅ lr correcto
    loss='binary_crossentropy',                # ✅
    metrics=['accuracy']
)

es = callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)
model.fit(
    X_train_res, y_train_res,
    epochs=200,
    validation_split=0.15,                     # ✅
    callbacks=[es]
)

# ✅ 5. Métricas correctas
y_prob = model.predict(X_test_prep).flatten()
y_pred = (y_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['No soporte', 'Soporte']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

---
---
# TARJETA DE REFERENCIA RÁPIDA

Para llevar en la cabeza al parcial.

---

## Lo más importante

### 1. Lee el caso → identifica el problema
```
¿Hay etiquetas?  → Supervisado / No supervisado
¿Salida numérica continua? → Regresión
¿2 clases?       → Binario:     sigmoid + binary_crossentropy
¿N clases?       → Multiclase:  softmax + sparse_categorical_crossentropy
¿Sin etiquetas?  → Clustering:  KMeans / DBSCAN / Jerárquico
```

### 2. Revisa la arquitectura
```
Capas ocultas → relu
Salida binaria → sigmoid (1 neurona)
Salida multiclase → softmax (N neuronas)
Salida regresión → sin activación (1 neurona)

CNN:
  Conv2D → MaxPooling → ... → FLATTEN → Dense → salida
  input_shape = (H, W, C)   ← siempre con el canal
  imágenes / 255.0           ← siempre normalizar

NLP:
  texto → Tokenizer → pad_sequences → Embedding → LSTM/Dense → salida
  ¡nunca texto crudo directamente a la red!

LSTM apilados:
  Todos menos el último → return_sequences=True
  El último → return_sequences=False
```

### 3. Revisa el pipeline de datos
```
ORDEN CORRECTO:
  1. train_test_split (con stratify=y si es clasificación)
  2. imputer.fit(X_train)  → imputer.transform(X_test)
  3. scaler.fit(X_train)   → scaler.transform(X_test)
  4. SMOTE.fit_resample(X_train, y_train)  (solo en train)
  5. tokenizer.fit_on_texts(X_train)  (solo en train)

VARIABLES:
  nominal (sin orden) → OneHotEncoder
  ordinal (con orden) → OrdinalEncoder con categories=[['a','b','c']]
  numérica con outliers → RobustScaler
  numérica normal → StandardScaler
  binaria (0/1) → no escalar
```

### 4. Revisa las métricas
```
Clasificación balanceada   → accuracy, F1
Clasificación desbalanceada → ROC-AUC, F1 macro, recall por clase
Regresión                  → MSE, MAE, R²
Clustering                 → silhouette, inercia
```

### 5. Los errores más tramposos
```
❌ Dense antes de Flatten en CNN
❌ input_shape=(28,28) sin el canal → debe ser (28,28,1)
❌ learning_rate > 0.1 → loss explota a NaN
❌ return_sequences=False en LSTM intermedio apilado
❌ accuracy en dataset 95/5
❌ SMOTE antes del split
❌ OrdinalEncoder en variable nominal
❌ scaler.fit(X_todo) antes del split
```